In [2]:
import requests
from bs4 import BeautifulSoup

r = requests.get("https://lrb.hawaii.gov/par/mission-history/")
soup = BeautifulSoup(r.text, "lxml")
print(soup.find("main") or soup.find("div", id="content"))

<main id="main" tabindex="-1">
<div class="gtranslate_wrapper" id="gt-wrapper-19809924"></div>
<article aria-label="article" class="post" role="contentinfo">
<header class="post-header" style="background-image: url('https://lrb.hawaii.gov/par/wp-content/uploads/sites/2/2019/11/par-label.jpg');">
<div class="post-header-container">
<div class="breadcrumbs" typeof="BreadcrumbList" vocab="http://schema.org/">
<span property="itemListElement" typeof="ListItem"><a class="home" href="https://lrb.hawaii.gov/par" property="item" title="Go to LRB - Public Access Room." typeof="WebPage"><span property="name">LRB - Public Access Room</span></a><meta content="1" property="position"/></span> &gt; <span class="post post-page current-item">Mission &amp; History</span> </div>
<h1 class="post-title">
		Mission &amp; History		</h1>
</div>
</header>
<hr class="separator-bar"/>
<div class="post-content">
<div class="post-content-container">
<div class="row">
<div class="col">
<h2 class="wp-block-heading">

In [4]:
"""
Hawaii LRB Combined Scraper
============================
Merges the best of both scripts:
  - SHA-256 deduplication (only re-downloads changed PDFs)
  - Polite random delays between requests
  - Per-page crash checkpoints
  - pdfplumber text extraction
  - RAG-ready .jsonl metadata output
  - Timestamped audit log file
  - Optional Selenium fallback for JS-rendered pages
"""

import hashlib
import json
import logging
import os
import random
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

# ── Optional Selenium (only imported if USE_SELENIUM = True) ──────────────────
USE_SELENIUM = False   # ← flip to True if the site needs JS rendering

if USE_SELENIUM:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.common.exceptions import TimeoutException
    from webdriver_manager.chrome import ChromeDriverManager

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper_audit.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)

# ── Config ────────────────────────────────────────────────────────────────────
TARGET_URLS = [
    "https://lrb.hawaii.gov/par/mission-history/",
    "https://lrb.hawaii.gov/par/current-legislature/",
    "https://lrb.hawaii.gov/par/hawaiis-legislature-and-government/hawaiis-legislative-branch/",
    "https://lrb.hawaii.gov/par/hawaiis-legislature-and-government/overview-of-branches-of-government/",
    "https://lrb.hawaii.gov/directory/",
]

OUTPUT_DIR   = Path("scraped_output")
PDF_DIR      = OUTPUT_DIR / "pdfs"
RESULTS_FILE = OUTPUT_DIR / "results.json"       # full page + PDF text dump
JSONL_FILE   = OUTPUT_DIR / "rag_metadata.jsonl" # one record per PDF, RAG-ready
CSV_LOG      = OUTPUT_DIR / "document_log.csv"   # dedup hash tracker

OUTPUT_DIR.mkdir(exist_ok=True)
PDF_DIR.mkdir(exist_ok=True)

MIN_DELAY       = 2.0   # seconds between page fetches
MAX_DELAY       = 5.0
PDF_MIN_DELAY   = 1.0   # shorter delay between PDF downloads
PDF_MAX_DELAY   = 2.5
REQUEST_TIMEOUT = 60

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 "
    "HawaiiLegResearch/1.0"
)

# ── Utilities ─────────────────────────────────────────────────────────────────

def polite_sleep(min_s: float = MIN_DELAY, max_s: float = MAX_DELAY) -> None:
    delay = random.uniform(min_s, max_s)
    log.debug("Sleeping %.1fs", delay)
    time.sleep(delay)


def sha256_of_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def load_hash_log() -> pd.DataFrame:
    if CSV_LOG.exists():
        df = pd.read_csv(CSV_LOG)
        log.info("Loaded existing hash log: %d records", len(df))
        return df
    log.info("No existing hash log found — starting fresh")
    return pd.DataFrame(columns=["url", "hash"])


def save_hash_log(df: pd.DataFrame) -> None:
    df.to_csv(CSV_LOG, index=False)


def is_changed(df: pd.DataFrame, url: str, new_hash: str) -> bool:
    """Return True if the URL is new or its content hash has changed."""
    row = df[df["url"] == url]
    return row.empty or row.iloc[0]["hash"] != new_hash


def update_hash_log(df: pd.DataFrame, url: str, new_hash: str) -> pd.DataFrame:
    if df[df["url"] == url].empty:
        df = pd.concat(
            [df, pd.DataFrame([{"url": url, "hash": new_hash}])],
            ignore_index=True,
        )
    else:
        df.loc[df["url"] == url, "hash"] = new_hash
    return df

# ── HTTP fetch (plain requests) ───────────────────────────────────────────────

def fetch_html_requests(url: str, session: requests.Session) -> str | None:
    try:
        log.info("GET (requests) %s", url)
        r = session.get(url, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r.text
    except Exception as e:
        log.error("Failed to fetch %s: %s", url, e)
        return None

# ── Selenium fetch (optional) ─────────────────────────────────────────────────

def build_driver(headless: bool = True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_argument(f"user-agent={USER_AGENT}")
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opts
    )
    driver.set_page_load_timeout(REQUEST_TIMEOUT)
    log.info("Selenium driver initialized (headless=%s)", headless)
    return driver


def fetch_html_selenium(url: str, driver) -> str | None:
    try:
        log.info("GET (selenium) %s", url)
        driver.get(url)
        WebDriverWait(driver, 20).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        polite_sleep()
        return driver.page_source
    except TimeoutException:
        log.warning("Timeout loading %s", url)
        return None
    except Exception as e:
        log.error("Selenium failed on %s: %s", url, e)
        return None

# ── HTML parsing ──────────────────────────────────────────────────────────────

def parse_page(html: str, base_url: str) -> dict:
    soup = BeautifulSoup(html, "lxml")

    title_tag = soup.find("title")
    title = title_tag.get_text(strip=True) if title_tag else ""

    for tag in soup(["nav", "footer", "script", "style", "header"]):
        tag.decompose()

    main = (
        soup.find("main")
        or soup.find("div", {"id": "content"})
        or soup.find("body")
    )
    text_content = main.get_text(separator="\n", strip=True) if main else ""

    pdf_links = []
    for a in soup.find_all("a", href=True):
        full = urljoin(base_url, a["href"])
        if full.lower().endswith(".pdf"):
            label = a.get_text(strip=True) or "Untitled"
            pdf_links.append({"url": full, "label": label})

    # Deduplicate by URL while preserving label
    seen = {}
    for item in pdf_links:
        seen.setdefault(item["url"], item)
    pdf_links = list(seen.values())

    return {"title": title, "text_content": text_content, "pdf_links": pdf_links}

# ── PDF download ──────────────────────────────────────────────────────────────

def download_pdf(
    pdf_url: str,
    label: str,
    session: requests.Session,
    hash_df: pd.DataFrame,
    run_time: str,
) -> tuple[dict | None, pd.DataFrame]:
    """
    Download a PDF only if it is new or its content has changed.
    Returns (rag_record | None, updated hash_df).
    """
    try:
        log.info("Fetching PDF: %s", pdf_url)
        r = session.get(pdf_url, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        content = r.content
        new_hash = sha256_of_bytes(content)

        if not is_changed(hash_df, pdf_url, new_hash):
            log.info("No change detected, skipping: %s", pdf_url)
            # Still return a record so the caller knows this PDF exists
            short = new_hash[:8]
            clean = "".join(c for c in label if c.isalnum() or c in " _")[:40].strip()
            local_path = PDF_DIR / f"{short}_{clean}.pdf"
            return _make_rag_record(pdf_url, label, local_path, new_hash, run_time, changed=False), hash_df

        short = new_hash[:8]
        clean = "".join(c for c in label if c.isalnum() or c in " _")[:40].strip()
        filename = f"{short}_{clean}.pdf"
        local_path = PDF_DIR / filename

        with open(local_path, "wb") as f:
            f.write(content)
        log.info("Saved PDF → %s (%.1f KB)", filename, local_path.stat().st_size / 1024)

        hash_df = update_hash_log(hash_df, pdf_url, new_hash)
        polite_sleep(PDF_MIN_DELAY, PDF_MAX_DELAY)

        return _make_rag_record(pdf_url, label, local_path, new_hash, run_time, changed=True), hash_df

    except Exception as e:
        log.error("PDF download failed %s: %s", pdf_url, e)
        return None, hash_df


def _make_rag_record(
    url: str, label: str, local_path: Path, content_hash: str, run_time: str, changed: bool
) -> dict:
    return {
        "url": url,
        "title": label,
        "local_path": str(local_path),
        "content_hash": content_hash,
        "last_synced": run_time,
        "changed": changed,
        "pdf_text": None,   # filled in after extraction
    }

# ── PDF text extraction ───────────────────────────────────────────────────────

def extract_pdf_text(local_path: Path) -> dict:
    result = {"page_count": 0, "full_text": "", "pages": [], "error": None}
    if not local_path.exists():
        result["error"] = "File not found"
        return result
    try:
        with pdfplumber.open(local_path) as pdf:
            result["page_count"] = len(pdf.pages)
            texts = []
            for i, page in enumerate(pdf.pages, 1):
                text = (page.extract_text() or "").strip()
                result["pages"].append({"page": i, "text": text})
                texts.append(text)
            result["full_text"] = "\n\n".join(texts).strip()
        log.info(
            "Extracted %d pages from %s", result["page_count"], local_path.name
        )
    except Exception as e:
        result["error"] = str(e)
        log.error("PDF extraction failed for %s: %s", local_path.name, e)
    return result

# ── Scrape one page ───────────────────────────────────────────────────────────

def scrape_page(
    url: str,
    session: requests.Session,
    hash_df: pd.DataFrame,
    run_time: str,
    driver=None,
) -> tuple[dict, pd.DataFrame]:

    record = {
        "url": url,
        "scraped_at": datetime.now().isoformat(),
        "title": "",
        "text_content": "",
        "pdf_links": [],
        "pdf_records": [],
        "error": None,
    }

    # Fetch HTML
    html = (
        fetch_html_selenium(url, driver)
        if USE_SELENIUM and driver
        else fetch_html_requests(url, session)
    )

    if not html:
        record["error"] = "Page load failed"
        return record, hash_df

    parsed = parse_page(html, url)
    record["title"]        = parsed["title"]
    record["text_content"] = parsed["text_content"]
    record["pdf_links"]    = [p["url"] for p in parsed["pdf_links"]]

    log.info("Found %d PDF links on %s", len(parsed["pdf_links"]), url)

    rag_records = []
    for pdf_item in parsed["pdf_links"]:
        rag_rec, hash_df = download_pdf(
            pdf_item["url"], pdf_item["label"], session, hash_df, run_time
        )
        if rag_rec is None:
            continue

        # Extract text (only worth doing if file exists)
        local = Path(rag_rec["local_path"])
        if local.exists():
            extraction = extract_pdf_text(local)
            rag_rec["pdf_text"]   = extraction["full_text"]
            rag_rec["page_count"] = extraction["page_count"]
            rag_rec["pages"]      = extraction["pages"]
            rag_rec["extract_error"] = extraction["error"]
        else:
            rag_rec["pdf_text"]      = None
            rag_rec["page_count"]    = 0
            rag_rec["extract_error"] = "File not on disk (unchanged, skipped re-download)"

        rag_rec["source_page"] = url
        rag_records.append(rag_rec)

    record["pdf_records"] = rag_records
    return record, hash_df

# ── Checkpoint helpers ────────────────────────────────────────────────────────

def save_results(all_results: list) -> None:
    with open(RESULTS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)
    log.info("Checkpoint saved → %s", RESULTS_FILE)


def _make_page_record(page_result: dict) -> dict:
    """Build a RAG-ready record from the scraped page text itself."""
    return {
        "type": "page",
        "url": page_result["url"],
        "title": page_result["title"],
        "local_path": None,
        "content_hash": sha256_of_bytes(page_result["text_content"].encode("utf-8")),
        "last_synced": page_result["scraped_at"],
        "changed": True,
        "pdf_text": None,
        "page_text": page_result["text_content"],
        "source_page": page_result["url"],
    }


def save_jsonl(all_results: list) -> None:
    records = []

    for page in all_results:
        # Always write a record for the page text itself
        if page.get("text_content"):
            records.append(_make_page_record(page))

        # Write a record for each PDF found on the page
        for rec in page.get("pdf_records", []):
            lean = {k: v for k, v in rec.items() if k != "pages"}
            lean["type"] = "pdf"
            records.append(lean)

    with open(JSONL_FILE, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    page_count = sum(1 for r in records if r["type"] == "page")
    pdf_count  = sum(1 for r in records if r["type"] == "pdf")
    log.info(
        "RAG metadata saved → %s (%d page records, %d PDF records)",
        JSONL_FILE, page_count, pdf_count,
    )

# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    log.info("═" * 60)
    log.info("Starting scrape — %d URLs", len(TARGET_URLS))
    log.info("Selenium mode: %s", USE_SELENIUM)

    run_time = datetime.now().isoformat()
    session  = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    hash_df     = load_hash_log()
    all_results = []
    driver      = build_driver() if USE_SELENIUM else None

    try:
        for i, url in enumerate(TARGET_URLS, 1):
            log.info("─── [%d/%d] %s", i, len(TARGET_URLS), url)

            record, hash_df = scrape_page(url, session, hash_df, run_time, driver)
            all_results.append(record)

            # Save everything after each page so a crash loses at most one page
            save_results(all_results)
            save_jsonl(all_results)
            save_hash_log(hash_df)

            if i < len(TARGET_URLS):
                polite_sleep()

    except KeyboardInterrupt:
        log.warning("Interrupted — partial results saved")

    finally:
        if driver:
            driver.quit()
            log.info("Selenium driver closed")

    # ── Summary ──────────────────────────────────────────────────────────────
    total_pdfs   = sum(len(r["pdf_links"]) for r in all_results)
    changed_pdfs = sum(
        1 for r in all_results
        for rec in r.get("pdf_records", [])
        if rec.get("changed")
    )
    errors = [r["url"] for r in all_results if r["error"]]

    log.info("═" * 60)
    log.info("Done.")
    log.info("  Pages scraped    : %d", len(all_results))
    log.info("  PDF links found  : %d", total_pdfs)
    log.info("  PDFs downloaded  : %d (new/changed)", changed_pdfs)
    log.info("  Errors           : %d %s", len(errors), errors or "")
    log.info("  Results JSON     : %s", RESULTS_FILE.resolve())
    log.info("  RAG metadata     : %s", JSONL_FILE.resolve())
    log.info("  Hash log         : %s", CSV_LOG.resolve())
    log.info("  PDFs folder      : %s", PDF_DIR.resolve())

    return all_results


if __name__ == "__main__":
    main()

2026-03-26 19:25:24,450 [INFO] ════════════════════════════════════════════════════════════
2026-03-26 19:25:24,451 [INFO] Starting scrape — 5 URLs
2026-03-26 19:25:24,451 [INFO] Selenium mode: False
2026-03-26 19:25:24,452 [INFO] No existing hash log found — starting fresh
2026-03-26 19:25:24,459 [INFO] ─── [1/5] https://lrb.hawaii.gov/par/mission-history/
2026-03-26 19:25:24,459 [INFO] GET (requests) https://lrb.hawaii.gov/par/mission-history/
2026-03-26 19:25:25,811 [INFO] Found 0 PDF links on https://lrb.hawaii.gov/par/mission-history/
2026-03-26 19:25:25,813 [INFO] Checkpoint saved → scraped_output/results.json
2026-03-26 19:25:25,815 [INFO] RAG metadata saved → scraped_output/rag_metadata.jsonl (1 page records, 0 PDF records)
2026-03-26 19:25:29,604 [INFO] ─── [2/5] https://lrb.hawaii.gov/par/current-legislature/
2026-03-26 19:25:29,606 [INFO] GET (requests) https://lrb.hawaii.gov/par/current-legislature/
2026-03-26 19:25:29,734 [INFO] Found 20 PDF links on https://lrb.hawaii.gov